# Explainable and Fair AI-Based Learning Risk Early Warning

This notebook supports an EI-style AI in education study using the Open University Learning Analytics Dataset (OULAD). It builds an early warning model, explains predictions, and audits fairness across student groups.

Main task: predict whether a student is at learning risk using only the first 4 weeks of course activity.

In [ ]:
from pathlib import Path
import zipfile
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
EARLY_DAYS = 28
DATA_DIR = Path('data/oulad')
FIG_DIR = Path('outputs/figures')
TABLE_DIR = Path('outputs/tables')
for path in [DATA_DIR, FIG_DIR, TABLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Dataset

Download OULAD from https://analyse.kmi.open.ac.uk/open-dataset and extract the CSV files into `data/oulad/`. The next cell can also try an automatic download.

In [ ]:
required_files = ['studentInfo.csv', 'studentVle.csv', 'studentAssessment.csv', 'assessments.csv', 'studentRegistration.csv', 'vle.csv', 'courses.csv']

def dataset_ready():
    return all((DATA_DIR / name).exists() for name in required_files)

def download_oulad():
    url = 'https://analyse.kmi.open.ac.uk/open-dataset/download'
    zip_path = DATA_DIR / 'oulad.zip'
    if not zip_path.exists():
        print('Downloading OULAD...')
        urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)

if not dataset_ready():
    try:
        download_oulad()
    except Exception as exc:
        print('Automatic download failed. Please download OULAD manually into:', DATA_DIR.resolve())
        print(exc)
else:
    print('Dataset is ready:', DATA_DIR.resolve())

In [ ]:
student_info = pd.read_csv(DATA_DIR / 'studentInfo.csv')
student_vle = pd.read_csv(DATA_DIR / 'studentVle.csv')
student_assessment = pd.read_csv(DATA_DIR / 'studentAssessment.csv')
assessments = pd.read_csv(DATA_DIR / 'assessments.csv')
registration = pd.read_csv(DATA_DIR / 'studentRegistration.csv')
vle = pd.read_csv(DATA_DIR / 'vle.csv')

for name, df in {'student_info': student_info, 'student_vle': student_vle, 'student_assessment': student_assessment, 'assessments': assessments, 'registration': registration, 'vle': vle}.items():
    print(f'{name:20s} {df.shape}')

student_info.head()

## Feature Engineering

Students with final results `Fail` or `Withdrawn` are treated as high-risk. Only learning activity from day 0 to day 28 is used.

In [ ]:
key_cols = ['code_module', 'code_presentation', 'id_student']
base = student_info.copy()
base['risk'] = base['final_result'].isin(['Fail', 'Withdrawn']).astype(int)

early_vle = student_vle[(student_vle['date'] >= 0) & (student_vle['date'] <= EARLY_DAYS)].copy()
early_vle = early_vle.merge(vle[['id_site', 'code_module', 'code_presentation', 'activity_type']], on=['id_site', 'code_module', 'code_presentation'], how='left')

vle_agg = early_vle.groupby(key_cols).agg(
    total_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique'),
    mean_clicks_per_event=('sum_click', 'mean'),
    max_clicks_per_event=('sum_click', 'max'),
    resource_count=('id_site', 'nunique'),
).reset_index()

activity = early_vle.pivot_table(index=key_cols, columns='activity_type', values='sum_click', aggfunc='sum', fill_value=0).reset_index()
activity.columns = [str(c).replace(' ', '_').lower() for c in activity.columns]

early_assess = student_assessment.merge(assessments, on='id_assessment', how='left')
early_assess = early_assess[(early_assess['date_submitted'] >= 0) & (early_assess['date_submitted'] <= EARLY_DAYS) & (early_assess['assessment_type'] != 'Exam')].copy()
early_assess['late_submission'] = (early_assess['date_submitted'] > early_assess['date']).astype(int)
early_assess['weighted_score'] = early_assess['score'] * early_assess['weight'].fillna(0) / 100
assess_agg = early_assess.groupby(key_cols).agg(
    submitted_assessments=('id_assessment', 'nunique'),
    mean_score=('score', 'mean'),
    min_score=('score', 'min'),
    weighted_score_sum=('weighted_score', 'sum'),
    late_submission_count=('late_submission', 'sum'),
).reset_index()

reg = registration.copy()
reg['registered_before_start'] = (reg['date_registration'] < 0).astype(int)
reg['early_unregistration'] = (reg['date_unregistration'].notna() & (reg['date_unregistration'] <= EARLY_DAYS)).astype(int)
reg = reg[key_cols + ['date_registration', 'registered_before_start', 'early_unregistration']]

features = base.merge(vle_agg, on=key_cols, how='left').merge(activity, on=key_cols, how='left').merge(assess_agg, on=key_cols, how='left').merge(reg, on=key_cols, how='left')
features = features.fillna({c: 0 for c in features.select_dtypes(include=np.number).columns})
features['clicks_per_active_day'] = features['total_clicks'] / features['active_days'].replace(0, np.nan)
features['clicks_per_active_day'] = features['clicks_per_active_day'].fillna(0)
features[['final_result', 'risk']].value_counts().sort_index()

## Model Training

Sensitive attributes are kept for fairness analysis, but excluded from model inputs by default.

In [ ]:
sensitive_cols = ['gender', 'age_band', 'disability', 'imd_band', 'region']
exclude_cols = key_cols + ['final_result', 'risk'] + sensitive_cols
X = features[[c for c in features.columns if c not in exclude_cols]].copy()
y = features['risk'].copy()
sensitive = features[sensitive_cols].copy()

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(X, y, sensitive, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])

clf = Pipeline([
    ('preprocess', preprocess),
    ('model', RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
])

clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]
pred = (proba >= 0.50).astype(int)

print(classification_report(y_test, pred, target_names=['not_risk', 'risk']))
print('AUC:', round(roc_auc_score(y_test, proba), 4), 'F1:', round(f1_score(y_test, pred), 4))

In [ ]:
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['not_risk', 'risk'], yticklabels=['not_risk', 'risk'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_matrix.png', dpi=300)
plt.show()

## Explainability

The cell first tries SHAP. If SHAP is unavailable, it uses permutation importance.

In [ ]:
def transformed_feature_names():
    names = list(numeric_features)
    if categorical_features:
        onehot = clf.named_steps['preprocess'].named_transformers_['cat'].named_steps['onehot']
        names += onehot.get_feature_names_out(categorical_features).tolist()
    return names

try:
    import shap
    Xt = clf.named_steps['preprocess'].transform(X_test)
    if hasattr(Xt, 'toarray'):
        Xt = Xt.toarray()
    sample_size = min(1000, Xt.shape[0])
    sample_idx = np.random.default_rng(RANDOM_STATE).choice(Xt.shape[0], sample_size, replace=False)
    explainer = shap.TreeExplainer(clf.named_steps['model'])
    shap_values = explainer.shap_values(Xt[sample_idx])
    values = shap_values[1] if isinstance(shap_values, list) else shap_values
    shap.summary_plot(values, Xt[sample_idx], feature_names=transformed_feature_names(), max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'shap_summary.png', dpi=300, bbox_inches='tight')
    plt.show()
except Exception as exc:
    print('SHAP failed, using permutation importance instead:', exc)
    perm = permutation_importance(clf, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
    importance = pd.DataFrame({'feature': X_test.columns, 'importance': perm.importances_mean}).sort_values('importance', ascending=False).head(20)
    sns.barplot(data=importance, x='importance', y='feature', color='#4C78A8')
    plt.title('Permutation Importance')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'permutation_importance.png', dpi=300)
    plt.show()

## Fairness Audit

The fairness table reports positive prediction rate, true positive rate, false positive rate, and precision for each group.

In [ ]:
def group_metrics(y_true, y_pred, group):
    rows = []
    frame = pd.DataFrame({'y_true': np.asarray(y_true), 'y_pred': np.asarray(y_pred), 'group': np.asarray(group)}).dropna()
    for value, part in frame.groupby('group'):
        tn, fp, fn, tp = confusion_matrix(part['y_true'], part['y_pred'], labels=[0, 1]).ravel()
        rows.append({
            'group': value,
            'n': len(part),
            'actual_risk_rate': part['y_true'].mean(),
            'predicted_risk_rate': part['y_pred'].mean(),
            'true_positive_rate': tp / (tp + fn) if (tp + fn) else np.nan,
            'false_positive_rate': fp / (fp + tn) if (fp + tn) else np.nan,
            'precision': tp / (tp + fp) if (tp + fp) else np.nan,
        })
    return pd.DataFrame(rows).sort_values('n', ascending=False)

fairness_tables = {}
for col in sensitive_cols:
    table = group_metrics(y_test, pred, s_test[col])
    fairness_tables[col] = table
    table.to_csv(TABLE_DIR / f'fairness_by_{col}.csv', index=False)
    print('\n===', col, '===')
    display(table)

In [ ]:
summary = []
for attr, table in fairness_tables.items():
    summary.append({
        'attribute': attr,
        'demographic_parity_gap': table['predicted_risk_rate'].max() - table['predicted_risk_rate'].min(),
        'equal_opportunity_gap': table['true_positive_rate'].max() - table['true_positive_rate'].min(),
        'false_positive_rate_gap': table['false_positive_rate'].max() - table['false_positive_rate'].min(),
    })
fairness_summary = pd.DataFrame(summary).sort_values('equal_opportunity_gap', ascending=False)
fairness_summary.to_csv(TABLE_DIR / 'fairness_summary.csv', index=False)
fairness_summary

## Paper Notes

Suggested research questions:

1. How accurately can early-course behavioral data predict student learning risk?
2. Which early learning behaviors contribute most to model predictions?
3. Does model performance differ across gender, disability, age, socioeconomic, or regional groups?
4. Can explainability and fairness auditing support a responsible human-in-the-loop early warning system?